Clone Locally

In [2]:
!git clone -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag 2>/dev/null || (git -C /content/ag fetch && git -C /content/ag checkout experiments-profiling && git -C /content/ag pull)
%cd /content/ag
!pip install -q -e .

/content/ag
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for alphagenome-pytorch (pyproject.toml) ... done


Mic test

In [3]:
!python -c "import alphagenome_pytorch; print('ok')"

ok


Setup

In [4]:
!python experiments/01_cache_weights.py


model_all_folds.safetensors: downloading bytes:  24% 218M/921M [00:03<00:05, 132MB/s, 18.4MB/s  ]   
model_all_folds.safetensors: downloading bytes:  26% 235M/921M [00:04<00:14, 48.5MB/s, 18.7MB/s  ]
model_all_folds.safetensors: downloading bytes:  55% 511M/921M [00:06<00:01, 241MB/s, 37.3MB/s  ]  ]
model_all_folds.safetensors: downloading bytes:  61% 559M/921M [00:06<00:01, 242MB/s, 41.3MB/s  ]  ]
model_all_folds.safetensors: reconstructing file:  51% 470M/921M [00:06<00:05, 85.8MB/s, 31.2MB/s  ]
model_all_folds.safetensors: downloading bytes:  64% 592M/921M [00:06<00:01, 211MB/s, 44.4MB/s  ] ] 
model_all_folds.safetensors: downloading bytes:  76% 698M/921M [00:07<00:01, 143MB/s, 50.5MB/s  ] ]
model_all_folds.safetensors: downloading bytes:  86% 790M/921M [00:18<00:26, 4.88MB/s, 24.5MB/s  ]]
model_all_folds.safetensors: downloading bytes:  88% 813M/921M [00:32<00:46, 2.30MB/s, 9.81MB/s  ]]
model_all_folds.safetensors: downloading bytes: 100% 813M/813M [00:32<00:00, 24.8MB/s, 9.70MB/s

16kb input Trace

In [5]:
!python experiments/02_trace_json.py --length 16384

sanity 16kb: 21 output tensors, non-finite: none
  atac.1                                        float32[1, 16384, 256]
  atac.128                                      float32[1, 128, 256]
  dnase.1                                       float32[1, 16384, 384]
  dnase.128                                     float32[1, 128, 384]
  procap.1                                      float32[1, 16384, 128]
  procap.128                                    float32[1, 128, 128]
  cage.1                                        float32[1, 16384, 640]
  cage.128                                      float32[1, 128, 640]
  rna_seq.1                                     float32[1, 16384, 768]
  rna_seq.128                                   float32[1, 128, 768]
  chip_tf.128                                   float32[1, 128, 1664]
  chip_histone.128                              float32[1, 128, 1152]
  contact_maps                                  float32[1, 8, 8, 28]
  splice_sites.logits                     

Readable view of the tree (depth=2)

In [6]:
!python experiments/03_trace_tree.py --depth 2

# tree_16kb.json  meta={"length": 16384, "device": "cuda", "dtype": "fp32", "python": "3.13.15", "torch": "2.11.0+cu130", "cuda": "13.0", "gpu": "NVIDIA A100-SXM4-80GB", "port_commit": "bd8895a6decd256664a378bc59cb09df8015d0ec", "upstream_base": "72268c0", "weights": "model_all_folds.safetensors", "driver": "580.82.07"}
<root> [AlphaGenome]  params=450.45M  in=[[float32[1, 16384, 4],int64[1]],{}] -> out={atac:{1:float32[1, 16384, 256],128:float32[1, 128, 256]},dnase:{1:...  [glue]
  encoder [SequenceEncoder]  params=89.97M  in=[[float32[1, 16384, 4]],{}] -> out=[float32[1, 1536, 128],{bin_size_1:float32[1, 768, 16384],bin_size_...  [S1_encoder]
    encoder.dna_embedder [DnaEmbedder]  params=3.00M  in=[[float32[1, 4, 16384]],{}] -> out=float32[1, 768, 16384]  [S1_encoder]
    encoder.pool [Pool1d] call 1/7  params=0  in=[[float32[1, 768, 16384]],{}] -> out=float32[1, 768, 8192]  [S1_encoder]
    encoder.down_blocks.0 [DownResBlock]  params=7.46M  in=[[float32[1, 768, 8192]],{}] -> out=f